In [1]:
import pandas as pd
import numpy as np
import os
import joblib
import shap
import matplotlib.pyplot as plt

from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    log_loss,
)

os.makedirs("results", exist_ok=True)
os.makedirs("models",  exist_ok=True)

c:\Users\Playdata\Desktop\SKN31-2nd-4Team\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv(r"./data/train_features.csv")
df.head()

,msno,is_churn,tenure_days,bd,age_bucket,gender_enc,city,registered_via,tenure_bucket,tx_count,...,active_days,completion_rate,skip_rate,listen_regularity,last_listen_date,days_since_last_listen,plays_recent30,plays_prev30,listen_trend_ratio,engagement_score
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,1,4347.0,36.0,36-45,1.0,18.0,9.0,초장기,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.300000
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,1,4346.0,38.0,36-45,0.0,10.0,9.0,초장기,1,...,521.0,0.910879,0.054948,13.880998,2017-02-15,14.0,3.0,NaN,NaN,5.857867
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,1,4154.0,27.0,26-35,1.0,11.0,9.0,초장기,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.300000
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,1,4137.0,23.0,19-25,1.0,13.0,9.0,초장기,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.300000
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,1,4081.0,27.0,26-35,0.0,3.0,9.0,초장기,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.300000


In [3]:
# 피처와 타겟 분리, 필요 없는 컬럼 제거
drop_cols = [
    "msno", 
    "is_churn",
    "last_tx_date",
    "last_expire_date",
    "last_listen_date",
    "age_bucket",
    "tenure_bucket",
]

feature_cols = [c for c in df.columns if c not in drop_cols] # 피처 컬럼 리스트, 컴프리헨션으로 drop_cols 제외

X = df[feature_cols].copy(), 
y = df["is_churn"].copy()

# 범주형은 Label Encoding, 수치형은 중앙값으로 채우기

for col in X.columns:
    if X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
    else:
        X[col] = X[col].fillna(X[col].median())

# 클래스 불균형(15:1) 보정 비율

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

# train / validation 분리
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# LightGBM 모델 생성

lgbm_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    verbose=-1,
)

# 학습
lgbm_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
)


# 예측
pred       = lgbm_model.predict(X_val)
pred_proba = lgbm_model.predict_proba(X_val)[:, 1]


# 평가
print("Confusion Matrix")
print(confusion_matrix(y_val, pred))

print("\nClassification Report")
print(classification_report(y_val, pred))

print("ROC-AUC :", roc_auc_score(y_val, pred_proba))
print("F1-score:", f1_score(y_val, pred))

AttributeError: 'tuple' object has no attribute 'columns'

In [ ]:
# 주요 평가지표

result_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-Score", "ROC-AUC"],
    "Score": [
        accuracy_score(y_val, pred),
        precision_score(y_val, pred),
        recall_score(y_val, pred),
        f1_score(y_val, pred),
        roc_auc_score(y_val, pred_proba),
    ],
})

result_df

In [ ]:
# 예측 확률 결과 확인

result = pd.DataFrame({
    "Actual":      y_val,
    "Predicted":   pred,
    "Probability": pred_proba,
})

result.head(20)

In [ ]:
# 피처 중요도 (Feature Importance) 시각화

# 중요도 추출
importance_df = pd.DataFrame({
    "Feature":    feature_cols,
    "Importance": lgbm_model.feature_importances_,
})

# 중요도 순 정렬 후 상위 10개 선택
top10 = (
    importance_df
    .sort_values("Importance", ascending=False)
    .head(10)
)

# 그래프
plt.figure(figsize=(10, 6))
plt.barh(top10["Feature"], top10["Importance"], color='cornflowerblue')
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importance (LightGBM)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("results/feature_importance_lgbm.png", dpi=300, bbox_inches='tight')
plt.show()

top10.head(20)

In [ ]:
# SHAP 피처 중요도 시각화

X_shap = X_val.sample(n=5000, random_state=42) # 5000개 샘플링

explainer   = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_shap, check_additivity=False)

# list 또는 array로 반환
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

shap_importance = pd.DataFrame({
    "feature":    feature_cols,
    "importance": np.abs(sv).mean(axis=0),
}).sort_values("importance", ascending=False)

top10_shap = shap_importance.head(10)

# 그래프
plt.figure(figsize=(10, 6))
plt.barh(top10_shap["feature"], top10_shap["importance"], color='cornflowerblue')
plt.xlabel("Mean |SHAP Value|")
plt.ylabel("Feature")
plt.title("Top 10 Feature Importance (SHAP) — LightGBM")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("results/shap_importance.png", dpi=300, bbox_inches='tight')
plt.show()

top10_shap

In [ ]:
# ROC Curve 시각화

# ROC Curve 계산
fpr, tpr, thresholds = roc_curve(y_val, pred_proba)

# AUC 계산
auc_score = roc_auc_score(y_val, pred_proba)

# 그래프
plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    color='dodgerblue',
    label=f"ROC Curve (AUC = {auc_score:.4f})",
)

# 시각화
plt.plot([0, 1], [0, 1], linestyle="--", color='black')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - LightGBM")
plt.legend(loc="lower right")

plt.grid(True)
plt.savefig("results/roc_lightgbm.png", dpi=300, bbox_inches='tight')
plt.show()

print("LightGBM ROC-AUC:", roc_auc_score(y_val, pred_proba))

In [ ]:
# "lightgbm_classification_report" 저장

report = classification_report(
    y_val,
    pred,
    output_dict=True,
)

report_df = pd.DataFrame(report).transpose()

report_df.to_csv(
    "results/lightgbm_classification_report.csv",
    encoding="utf-8-sig",
)

print("저장 완료: results/lightgbm_classification_report.csv")

In [ ]:
# feature 중요도 저장

# feature_importances_ 기반 전체 저장
importance_df.sort_values("Importance", ascending=False).to_csv(
    "results/importance_lightgbm.csv",
    index=False,
    encoding="utf-8-sig",
)

# SHAP 기반 전체 저장
shap_importance.to_csv(
    "results/shap_importance.csv",
    index=False,
    encoding="utf-8-sig",
)

print("저장 완료: results/importance_lightgbm.csv")
print("저장 완료: results/shap_importance.csv")

In [ ]:
# 모델 저장

joblib.dump(lgbm_model, "./models/lightgbm.pkl")

print("모델 저장 완료: ./models/lightgbm.pkl")

In [ ]:
# 모델성능 비교

# 성능 지표 계산
logloss = log_loss(y_val, pred_proba)
auc     = roc_auc_score(y_val, pred_proba)
f1      = f1_score(y_val, pred)

# DataFrame 생성 (Model 이름 지정)
result_perf = pd.DataFrame({
    "Model":    ["LightGBM"],
    "Log Loss": [logloss],
    "AUC":      [auc],
    "F1":       [f1],
})

# 폴더 및 파일 경로 설정
os.makedirs("results", exist_ok=True)
save_path = "results/model_performance.csv"

# CSV 파일로 저장 (기존 파일이 있으면 이어서 추가, 없으면 새로 작성)
if os.path.exists(save_path):
    result_perf.to_csv(save_path, mode='a', header=False, index=False, encoding="utf-8-sig")
    print(f"기존 {save_path} 파일에 LightGBM 결과가 추가되었습니다.")
else:
    result_perf.to_csv(save_path, mode='w', header=True,  index=False, encoding="utf-8-sig")
    print(f"새로운 {save_path} 파일로 저장되었습니다.")

# 저장된 데이터 확인
result_perf

In [ ]:
result = pd.DataFrame({
    "Actual":      y_val,
    "Predicted":   pred,
    "Probability": pred_proba,
})

result["risk_group"] = pd.cut(
    result["Probability"],
    bins=[0, 0.25, 0.50, 0.75, 1.0],
    labels=["낮음", "보통", "높음", "매우높음"]
)

result["risk_group"].value_counts()

NameError: name 'pd' is not defined